# State-Conditioned Actor Selector v1

Direction: `model_selection` diagnostic.  
Role: `diagnose`; no runtime promotion from this notebook.  
Changed layer: model-selection/composition among already exported actor daily paths; no new training and no shared-code change.

## Governance

- Canonical baseline: current scheduled `balanced_rule_residual` family and the governed runtime composer line.
- Direct comparison: `upstream_actor_low_risk_refresh_v1`, which showed no current actor by itself clears the `+30pp / <=15%` strict target.
- Recent failure to avoid duplicating: `current_coverage_rebound_downtrend_router_v1` improved the 2026 rebound cluster but failed strict and walk-forward drawdown through a path-level threshold router.
- Target windows: strict OOS through current coverage plus saved `wf_2020`, `wf_2022`, `wf_2023_2024` actor exports.
- Accepted trade-off: this run may identify a diagnostic selector, but promotion requires strict `relative_return >= +30pp`, strict `max_dd <= 15%`, all walk-forward MaxDD `<=15%`, and walk-forward relative-return floors.
- Comparison-table row key: `state_conditioned_actor_selector_v1`.

## Historical reflection

- Closest success: `predictive_drawdown_risk_gate_v1` cleared the stale strict target and `wf_2022`, but missed walk-forward MaxDD in `wf_2020`/`wf_2023_2024` and lacked current full-coverage actor evidence.
- Closest failure: `upstream_actor_low_risk_refresh_v1` showed refreshed actors as-is have no strict/full pass; `current_coverage_rebound_downtrend_router_v1` showed direct path-level rebound/risk thresholds are not enough.
- Why this is not a duplicate: this notebook does not cap positions or search another path-level threshold. It selects among existing actor outputs by state, testing whether the missing layer is model selection/composition rather than actor training.

## External basis

- Probability of Backtest Overfitting: repeated selection over many configurations needs a compact grid and explicit OOS/WF gate. https://doi.org/10.21314/JCF.2016.322
- Model Confidence Set / model-picking overfit: model selection itself can overfit and must be tested out of sample. https://doi.org/10.3233/AF-180231
- Deflated Sharpe Ratio: multiple testing correction should inform model-selection evidence. https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf
- Qlib: keep the research loop inside a reproducible AI quant workflow with model training, backtesting, and RL support. https://github.com/microsoft/qlib

## Hypothesis

If existing actors already contain the necessary behaviors, a compact state-conditioned selector should combine a defensive actor in high-risk states and a participation actor in high-volatility rebound states while preserving walk-forward controls. If it still fails, the next step should be training-time state-conditioned objectives or actor refresh rather than more model-selection thresholds.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root()
EXPERIMENT = "state_conditioned_actor_selector_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime/model_selection/card/upstream_actor_low_risk_refresh_v1"
EXPORT_DIR = SOURCE_CARD / "exports"
SUMMARY_PATH = SOURCE_CARD / "actor_family_summary.csv"
PANEL_PATH = ROOT / "data/processed/panels/soxl_soxx_vix_full.csv"

STARTING_NAV = 1_000_000.0
STRICT_TARGET_REL = 30.0
STRICT_TARGET_DD = 15.0
WF_REL_FLOORS = {"wf_2020": -10.0, "wf_2022": 30.0, "wf_2023_2024": 30.0}
WINDOWS = ["strict_oos", "wf_2020", "wf_2022", "wf_2023_2024"]
POOR_CLUSTERS = {
    "cluster_2025_high_vol_rebound": ("2025-04-16", "2025-07-17"),
    "cluster_2026_high_vol_rebound": ("2026-02-17", "2026-05-15"),
}

for path in [SUMMARY_PATH, PANEL_PATH, EXPORT_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"repo={ROOT}")
print(f"card={CARD.relative_to(ROOT)}")


repo=/Users/rziiiii/.openclaw/workspace
card=runtime/model_selection/card/state_conditioned_actor_selector_v1


In [2]:
def load_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    required = {"datetime", "price", "position"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.relative_to(ROOT)} missing {missing}")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["position"] = pd.to_numeric(df["position"], errors="coerce").clip(0.0, 1.0)
    return df.dropna(subset=["price", "position"]).reset_index(drop=True)


def load_semis_panel(path: Path) -> pd.DataFrame:
    panel = pd.read_csv(path)
    panel["datetime"] = pd.to_datetime(panel["date"])
    panel = panel.sort_values("datetime").reset_index(drop=True)
    soxl = pd.to_numeric(panel["soxl_close"], errors="coerce")
    soxx = pd.to_numeric(panel["soxx_close"], errors="coerce")
    vix = pd.to_numeric(panel["vix"], errors="coerce")
    ratio = soxl / soxx.replace(0.0, np.nan)
    ret = soxl.pct_change()
    rv21 = ret.rolling(21, min_periods=5).std() * np.sqrt(252.0)
    ratio_mean = ratio.rolling(252, min_periods=63).mean()
    ratio_std = ratio.rolling(252, min_periods=63).std()
    out = pd.DataFrame({"datetime": panel["datetime"]})
    out["soxl_mom_10"] = soxl / soxl.shift(10) - 1.0
    out["soxl_mom_21"] = soxl / soxl.shift(21) - 1.0
    out["soxl_drawdown_63"] = soxl / soxl.rolling(63, min_periods=2).max() - 1.0
    out["realized_vol_21"] = rv21
    out["realized_vol_pct_21"] = rv21.rolling(252, min_periods=63).rank(pct=True)
    out["vol_of_vol_21"] = rv21.rolling(21, min_periods=5).std()
    out["vix_percentile_252"] = vix.rolling(252, min_periods=63).rank(pct=True)
    out["soxx_mom_21"] = soxx / soxx.shift(21) - 1.0
    out["soxl_soxx_ratio_z_252"] = (ratio - ratio_mean) / ratio_std.replace(0.0, np.nan)
    out["soxl_soxx_ratio_widen_21"] = out["soxl_soxx_ratio_z_252"] - out["soxl_soxx_ratio_z_252"].shift(21)
    return out


SEMIS_PANEL = load_semis_panel(PANEL_PATH)


def recompute_nav(price: np.ndarray, position: np.ndarray) -> np.ndarray:
    returns = np.zeros_like(price, dtype=float)
    returns[1:] = price[1:] / price[:-1] - 1.0
    applied_position = np.r_[position[0], position[:-1]]
    return STARTING_NAV * np.cumprod(1.0 + applied_position * returns)


def metrics_from_position(price: np.ndarray, position: np.ndarray) -> dict[str, float]:
    nav = recompute_nav(price, position)
    buyhold = STARTING_NAV * price / price[0]
    returns = price[1:] / price[:-1] - 1.0
    strat_returns = position[:-1] * returns if len(position) > 1 else np.array([], dtype=float)
    sharpe = float(np.mean(strat_returns) / np.std(strat_returns) * np.sqrt(252.0)) if len(strat_returns) > 2 and np.std(strat_returns) > 0 else 0.0
    return {
        "return": float((nav[-1] / nav[0] - 1.0) * 100.0),
        "buy_and_hold_return": float((buyhold[-1] / buyhold[0] - 1.0) * 100.0),
        "relative_return": float(((nav[-1] / nav[0]) - (buyhold[-1] / buyhold[0])) * 100.0),
        "sharpe": sharpe,
        "max_dd": float((1.0 - nav / np.maximum.accumulate(nav)).max() * 100.0),
        "avg_pos": float(np.nanmean(position)),
        "turnover": float(np.abs(np.diff(position, prepend=position[0])).sum()),
        "rows": int(len(position)),
    }


def sub_metrics(frame: pd.DataFrame, position: np.ndarray, start: str, end: str) -> dict[str, float]:
    dt = frame["datetime"]
    mask = (dt >= pd.Timestamp(start)) & (dt <= pd.Timestamp(end))
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) < 2:
        return {"return": np.nan, "buy_and_hold_return": np.nan, "relative_return": np.nan, "sharpe": np.nan, "max_dd": np.nan, "avg_pos": np.nan, "turnover": np.nan, "rows": int(len(idx))}
    return metrics_from_position(frame["price"].to_numpy(dtype=float)[idx], position[idx])


def build_window_frames(profile_names: list[str]) -> dict[str, pd.DataFrame]:
    window_frames = {}
    for window in WINDOWS:
        base = None
        for profile in profile_names:
            path = EXPORT_DIR / f"{profile}__{window}.csv"
            daily = load_daily(path)[["datetime", "price", "position"]].rename(columns={"position": profile})
            if base is None:
                base = daily
            else:
                base = base.merge(daily[["datetime", profile]], on="datetime", how="inner", validate="one_to_one")
        if base is None or base.empty:
            raise ValueError(f"empty window {window}")
        base = base.merge(SEMIS_PANEL, on="datetime", how="left")
        feature_cols = [
            "soxl_mom_10", "soxl_mom_21", "soxl_drawdown_63", "realized_vol_21",
            "realized_vol_pct_21", "vol_of_vol_21", "vix_percentile_252", "soxx_mom_21",
            "soxl_soxx_ratio_z_252", "soxl_soxx_ratio_widen_21",
        ]
        for col in feature_cols:
            base[col] = pd.to_numeric(base[col], errors="coerce").fillna(0.0)
        window_frames[window] = base.sort_values("datetime").reset_index(drop=True)
    return window_frames


def state_masks(frame: pd.DataFrame, cfg: dict) -> tuple[np.ndarray, np.ndarray, dict[str, np.ndarray]]:
    high_vol = frame["realized_vol_pct_21"].to_numpy(dtype=float) >= cfg["rebound_vol_pct"]
    rebound = (
        high_vol
        & (frame["soxl_mom_21"].to_numpy(dtype=float) >= cfg["rebound_mom21"])
        & (frame["soxl_mom_10"].to_numpy(dtype=float) >= cfg["rebound_mom10"])
        & (frame["soxl_drawdown_63"].to_numpy(dtype=float) >= cfg["rebound_min_dd63"])
    )
    if cfg["require_soxx_positive"]:
        rebound = rebound & (frame["soxx_mom_21"].to_numpy(dtype=float) >= 0.0)
    downtrend = high_vol & ((frame["soxl_drawdown_63"].to_numpy(dtype=float) <= cfg["risk_dd63"]) | (frame["soxl_mom_21"].to_numpy(dtype=float) <= cfg["risk_mom21"]))
    vix_stress = (frame["vix_percentile_252"].to_numpy(dtype=float) >= cfg["risk_vix_pct"]) & (frame["soxl_mom_10"].to_numpy(dtype=float) <= 0.0)
    semis_stress = ((frame["soxl_soxx_ratio_z_252"].to_numpy(dtype=float) <= -0.75) | (frame["soxl_soxx_ratio_widen_21"].to_numpy(dtype=float) <= -0.05)) & (frame["soxx_mom_21"].to_numpy(dtype=float) <= 0.0)
    risk = downtrend | vix_stress | (semis_stress if cfg["use_semis_stress"] else False)
    return rebound, risk, {"high_vol": high_vol, "downtrend": downtrend, "vix_stress": vix_stress, "semis_stress": semis_stress}


def selector_position(frame: pd.DataFrame, cfg: dict) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    rebound, risk, states = state_masks(frame, cfg)
    base = frame[cfg["base_actor"]].to_numpy(dtype=float)
    risk_pos = frame[cfg["risk_actor"]].to_numpy(dtype=float)
    rebound_pos = frame[cfg["rebound_actor"]].to_numpy(dtype=float)
    position = base.copy()
    position[risk] = risk_pos[risk]
    position[rebound & ~risk] = rebound_pos[rebound & ~risk]
    states.update({"rebound": rebound, "risk": risk})
    return np.clip(position, 0.0, 1.0), states


def score_candidate(metrics_by_window: dict[str, dict], cluster_metrics: dict[str, dict], base_cluster: dict[str, dict]) -> tuple[float, bool, bool, bool]:
    strict = metrics_by_window["strict_oos"]
    strict_pass = strict["relative_return"] >= STRICT_TARGET_REL and strict["max_dd"] <= STRICT_TARGET_DD
    wf_pass = True
    wf_loss = 0.0
    for wf, floor in WF_REL_FLOORS.items():
        m = metrics_by_window[wf]
        wf_pass = wf_pass and (m["max_dd"] <= STRICT_TARGET_DD) and (m["relative_return"] >= floor)
        wf_loss += 2.0 * max(0.0, m["max_dd"] - STRICT_TARGET_DD) ** 2
        wf_loss += 0.25 * max(0.0, floor - m["relative_return"]) ** 2
    strict_loss = max(0.0, STRICT_TARGET_REL - strict["relative_return"]) ** 2 + 4.0 * max(0.0, strict["max_dd"] - STRICT_TARGET_DD) ** 2
    cluster_pass = True
    cluster_loss = 0.0
    for name, m in cluster_metrics.items():
        delta = m["relative_return"] - base_cluster[name]["relative_return"]
        cluster_pass = cluster_pass and (delta >= 0.0) and (m["max_dd"] <= STRICT_TARGET_DD)
        cluster_loss += 0.10 * max(0.0, 20.0 - delta) ** 2
        cluster_loss += 2.0 * max(0.0, m["max_dd"] - STRICT_TARGET_DD) ** 2
    turnover_loss = 0.02 * strict["turnover"]
    return float(strict_loss + wf_loss + cluster_loss + turnover_loss), bool(strict_pass), bool(strict_pass and wf_pass), bool(strict_pass and wf_pass and cluster_pass)


In [3]:
summary_df = pd.read_csv(SUMMARY_PATH)
wf_profiles = summary_df.loc[summary_df["has_all_wf"].fillna(False).astype(bool), "profile"].tolist()
# Use the compact all-WF actor pool to avoid selecting strict-only actors that cannot prove generalization.
profile_names = sorted(set(wf_profiles))
window_frames = build_window_frames(profile_names)

actor_pool_rows = []
for profile in profile_names:
    for window, frame in window_frames.items():
        metrics = metrics_from_position(frame["price"].to_numpy(dtype=float), frame[profile].to_numpy(dtype=float))
        actor_pool_rows.append({"profile": profile, "window": window, **metrics, "start": frame["datetime"].iloc[0].date().isoformat(), "end": frame["datetime"].iloc[-1].date().isoformat()})
actor_pool_df = pd.DataFrame(actor_pool_rows)
actor_pool_df.to_csv(CARD / "actor_pool_metrics.csv", index=False)

display(Markdown("## Actor pool"))
display(actor_pool_df)
print(f"profile_count={len(profile_names)} profiles={profile_names}")


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_19346/2892674326.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  wf_profiles = summary_df.loc[summary_df["has_all_wf"].fillna(False).astype(bool), "profile"].tolist()


## Actor pool

,profile,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,turnover,rows,start,end
0,balanced,strict_oos,1015.809770,816.512500,199.297271,0.952213,55.159576,0.494162,53.711446,1605,2020-01-02,2026-05-21
1,balanced,wf_2020,37.892450,58.093571,-20.201121,0.818988,50.511821,0.432220,5.916296,252,2020-01-02,2020-12-30
2,balanced,wf_2022,-44.537180,-86.588072,42.050892,-1.355817,46.447969,0.243205,3.872981,250,2022-01-03,2022-12-29
3,balanced,wf_2023_2024,195.302339,198.504290,-3.201952,1.398723,32.634108,0.482117,12.065274,501,2023-01-03,2024-12-30
4,balanced_rule_residual_imitation_v4,strict_oos,1797.494733,816.512500,980.982234,1.129959,48.006757,0.475650,32.346981,1605,2020-01-02,2026-05-21
5,balanced_rule_residual_imitation_v4,wf_2020,69.448916,58.093571,11.355346,1.186122,28.638902,0.387232,3.895391,252,2020-01-02,2020-12-30
6,balanced_rule_residual_imitation_v4,wf_2022,-46.018362,-86.588072,40.569709,-1.431572,47.752097,0.255771,3.856705,250,2022-01-03,2022-12-29
7,balanced_rule_residual_imitation_v4,wf_2023_2024,225.667128,198.504290,27.162838,1.514954,31.630825,0.480648,9.847341,501,2023-01-03,2024-12-30
8,balanced_rule_residual_mode_v1,strict_oos,2356.301891,816.512500,1539.789392,1.198294,48.735024,0.474314,33.488324,1605,2020-01-02,2026-05-21
9,balanced_rule_residual_mode_v1,wf_2020,69.448916,58.093571,11.355346,1.186122,28.638902,0.387232,3.895391,252,2020-01-02,2020-12-30


profile_count=6 profiles=['balanced', 'balanced_rule_residual_imitation_v4', 'balanced_rule_residual_mode_v1', 'balanced_rule_residual_v19_reg', 'balanced_rule_residual_v19_reg_light', 'balanced_rule_residual_v22']


In [4]:
GRID = {
    "base_actor": profile_names,
    "risk_actor": profile_names,
    "rebound_actor": profile_names,
    "rebound_mom21": [0.25, 0.40],
    "rebound_mom10": [0.02],
    "rebound_vol_pct": [0.65, 0.80],
    "rebound_min_dd63": [-0.25, -0.15],
    "require_soxx_positive": [False, True],
    "risk_dd63": [-0.08, -0.15],
    "risk_mom21": [0.0],
    "risk_vix_pct": [0.60, 0.75],
    "use_semis_stress": [False, True],
}


def iter_grid():
    keys = list(GRID)
    for values in product(*(GRID[k] for k in keys)):
        yield dict(zip(keys, values))


def add_metrics(row: dict, prefix: str, metrics: dict) -> None:
    for k, v in metrics.items():
        row[f"{prefix}_{k}"] = v

rows = []
candidate_id = 0
strict_frame = window_frames["strict_oos"]
base_cluster_cache: dict[str, dict[str, dict]] = {}
for base_actor in profile_names:
    base_pos = strict_frame[base_actor].to_numpy(dtype=float)
    base_cluster_cache[base_actor] = {name: sub_metrics(strict_frame, base_pos, start, end) for name, (start, end) in POOR_CLUSTERS.items()}

for cfg in iter_grid():
    metrics_by_window = {}
    state_counts = {}
    positions = {}
    for window, frame in window_frames.items():
        position, states = selector_position(frame, cfg)
        positions[window] = position
        metrics_by_window[window] = metrics_from_position(frame["price"].to_numpy(dtype=float), position)
        state_counts[window] = {f"{name}_days": int(np.sum(mask)) for name, mask in states.items()}
        state_counts[window]["changed_vs_base_days"] = int(np.sum(np.abs(position - frame[cfg["base_actor"]].to_numpy(dtype=float)) > 1e-9))
    cluster_metrics = {name: sub_metrics(strict_frame, positions["strict_oos"], start, end) for name, (start, end) in POOR_CLUSTERS.items()}
    base_cluster = base_cluster_cache[cfg["base_actor"]]
    robust_score, strict_pass, wf_pass, full_pass = score_candidate(metrics_by_window, cluster_metrics, base_cluster)
    row = {"candidate_id": candidate_id, **cfg, "robust_score": robust_score, "strict_pass": strict_pass, "wf_pass": wf_pass, "full_pass": full_pass}
    for window in WINDOWS:
        add_metrics(row, window, metrics_by_window[window])
        for k, v in state_counts[window].items():
            row[f"{window}_{k}"] = v
    for name, m in cluster_metrics.items():
        add_metrics(row, name, m)
        row[f"{name}_excess_delta_vs_base_actor"] = m["relative_return"] - base_cluster[name]["relative_return"]
        row[f"{name}_return_delta_vs_base_actor"] = m["return"] - base_cluster[name]["return"]
    rows.append(row)
    candidate_id += 1

screen_df = pd.DataFrame(rows).sort_values(["full_pass", "wf_pass", "strict_pass", "robust_score"], ascending=[False, False, False, True]).reset_index(drop=True)
screen_df.to_csv(CARD / "actor_selector_screen.csv", index=False)
screen_df.head(100).to_csv(CARD / "actor_selector_top_configs.csv", index=False)
changed_screen_df = screen_df.loc[screen_df["strict_oos_changed_vs_base_days"] > 0].copy()
changed_screen_df.head(100).to_csv(CARD / "actor_selector_top_changed_configs.csv", index=False)
family_summary = (
    screen_df.groupby(["base_actor", "risk_actor", "rebound_actor"])
    .agg(
        candidates=("candidate_id", "count"),
        strict_pass_count=("strict_pass", "sum"),
        wf_pass_count=("wf_pass", "sum"),
        full_pass_count=("full_pass", "sum"),
        best_score=("robust_score", "min"),
        best_strict_relative=("strict_oos_relative_return", "max"),
        best_strict_max_dd=("strict_oos_max_dd", "min"),
        best_2025_delta=("cluster_2025_high_vol_rebound_excess_delta_vs_base_actor", "max"),
        best_2026_delta=("cluster_2026_high_vol_rebound_excess_delta_vs_base_actor", "max"),
    )
    .reset_index()
    .sort_values(["full_pass_count", "wf_pass_count", "best_score"], ascending=[False, False, True])
)
family_summary.to_csv(CARD / "actor_selector_family_summary.csv", index=False)

print(f"screened_candidates={len(screen_df)}")
print(f"strict_pass_count={int(screen_df['strict_pass'].sum())}")
print(f"wf_pass_count={int(screen_df['wf_pass'].sum())}")
print(f"full_pass_count={int(screen_df['full_pass'].sum())}")
display(Markdown("## Top actor selectors"))
display(screen_df.head(12))


screened_candidates=27648
strict_pass_count=0
wf_pass_count=0
full_pass_count=0


## Top actor selectors

,candidate_id,base_actor,risk_actor,rebound_actor,rebound_mom21,rebound_mom10,rebound_vol_pct,rebound_min_dd63,require_soxx_positive,risk_dd63,risk_mom21,risk_vix_pct,use_semis_stress,robust_score,strict_pass,wf_pass,full_pass,strict_oos_return,strict_oos_buy_and_hold_return,strict_oos_relative_return,strict_oos_sharpe,strict_oos_max_dd,strict_oos_avg_pos,strict_oos_turnover,strict_oos_rows,strict_oos_high_vol_days,strict_oos_downtrend_days,strict_oos_vix_stress_days,strict_oos_semis_stress_days,strict_oos_rebound_days,strict_oos_risk_days,strict_oos_changed_vs_base_days,wf_2020_return,wf_2020_buy_and_hold_return,wf_2020_relative_return,wf_2020_sharpe,wf_2020_max_dd,wf_2020_avg_pos,wf_2020_turnover,wf_2020_rows,wf_2020_high_vol_days,wf_2020_downtrend_days,wf_2020_vix_stress_days,wf_2020_semis_stress_days,wf_2020_rebound_days,wf_2020_risk_days,wf_2020_changed_vs_base_days,wf_2022_return,wf_2022_buy_and_hold_return,wf_2022_relative_return,wf_2022_sharpe,wf_2022_max_dd,wf_2022_avg_pos,wf_2022_turnover,wf_2022_rows,wf_2022_high_vol_days,wf_2022_downtrend_days,wf_2022_vix_stress_days,wf_2022_semis_stress_days,wf_2022_rebound_days,wf_2022_risk_days,wf_2022_changed_vs_base_days,wf_2023_2024_return,wf_2023_2024_buy_and_hold_return,wf_2023_2024_relative_return,wf_2023_2024_sharpe,wf_2023_2024_max_dd,wf_2023_2024_avg_pos,wf_2023_2024_turnover,wf_2023_2024_rows,wf_2023_2024_high_vol_days,wf_2023_2024_downtrend_days,wf_2023_2024_vix_stress_days,wf_2023_2024_semis_stress_days,wf_2023_2024_rebound_days,wf_2023_2024_risk_days,wf_2023_2024_changed_vs_base_days,cluster_2025_high_vol_rebound_return,cluster_2025_high_vol_rebound_buy_and_hold_return,cluster_2025_high_vol_rebound_relative_return,cluster_2025_high_vol_rebound_sharpe,cluster_2025_high_vol_rebound_max_dd,cluster_2025_high_vol_rebound_avg_pos,cluster_2025_high_vol_rebound_turnover,cluster_2025_high_vol_rebound_rows,cluster_2025_high_vol_rebound_excess_delta_vs_base_actor,cluster_2025_high_vol_rebound_return_delta_vs_base_actor,cluster_2026_high_vol_rebound_return,cluster_2026_high_vol_rebound_buy_and_hold_return,cluster_2026_high_vol_rebound_relative_return,cluster_2026_high_vol_rebound_sharpe,cluster_2026_high_vol_rebound_max_dd,cluster_2026_high_vol_rebound_avg_pos,cluster_2026_high_vol_rebound_turnover,cluster_2026_high_vol_rebound_rows,cluster_2026_high_vol_rebound_excess_delta_vs_base_actor,cluster_2026_high_vol_rebound_return_delta_vs_base_actor
0,16512,balanced_rule_residual_v19_reg,balanced_rule_residual_v19_reg,balanced_rule_residual_v19_reg,0.25,0.02,0.65,-0.25,False,-0.08,0.0,0.60,False,6645.014884,False,False,False,1667.806381,816.5125,851.293881,1.151077,43.53497,0.449886,31.17529,1605,618,525,386,585,68,671,0,69.448916,58.093571,11.355346,1.186122,28.638902,0.387232,3.895391,252,119,99,60,58,12,119,0,-47.263003,-86.588072,39.325069,-1.356866,49.774907,0.273316,4.082471,250,156,153,128,159,4,190,0,218.937254,198.50429,20.432964,1.51699,30.714702,0.46858,9.652261,501,127,116,87,192,11,159,0,80.099055,192.505345,-112.40629,5.464897,10.131948,0.563993,0.823438,63,0.0,0.0,4.021956,155.13597,-151.114014,2.152755,1.119098,0.048717,0.702077,63,0.0,0.0
1,16513,balanced_rule_residual_v19_reg,balanced_rule_residual_v19_reg,balanced_rule_residual_v19_reg,0.25,0.02,0.65,-0.25,False,-0.08,0.0,0.60,True,6645.014884,False,False,False,1667.806381,816.5125,851.293881,1.151077,43.53497,0.449886,31.17529,1605,618,525,386,585,68,840,0,69.448916,58.093571,11.355346,1.186122,28.638902,0.387232,3.895391,252,119,99,60,58,12,120,0,-47.263003,-86.588072,39.325069,-1.356866,49.774907,0.273316,4.082471,250,156,153,128,159,4,212,0,218.937254,198.50429,20.432964,1.51699,30.714702,0.46858,9.652261,501,127,116,87,192,11,253,0,80.099055,192.505345,-112.40629,5.464897,10.131948,0.563993,0.823438,63,0.0,0.0,4.021956,155.13597,-151.114014,2.152755,1.119098,0.048717,0.702077,63,0.0,0.0
2,16514,balanced_rule_residual_v19_reg,balanced_rule_residual_v19_reg,balanced_rule_residual_v19_reg,0.25,0.02,0.65,-0.25,Fa

In [5]:
selected = screen_df.iloc[0].to_dict()
selected_cfg = {k: selected[k] for k in GRID.keys()}
selected_candidate_id = int(selected["candidate_id"])

selected_metrics_rows = []
for window, frame in window_frames.items():
    position, states = selector_position(frame, selected_cfg)
    metrics = metrics_from_position(frame["price"].to_numpy(dtype=float), position)
    selected_metrics_rows.append({"window": window, **metrics, **{f"{name}_days": int(np.sum(mask)) for name, mask in states.items()}, "changed_vs_base_days": int(np.sum(np.abs(position - frame[selected_cfg["base_actor"]].to_numpy(dtype=float)) > 1e-9))})
    nav = recompute_nav(frame["price"].to_numpy(dtype=float), position)
    buyhold = STARTING_NAV * frame["price"].to_numpy(dtype=float) / frame["price"].iloc[0]
    out = pd.DataFrame({
        "datetime": frame["datetime"].dt.strftime("%Y-%m-%d"),
        "price": frame["price"],
        "strategy_nav": nav,
        "buyhold_nav": buyhold,
        "position": position,
        "base_actor": selected_cfg["base_actor"],
        "risk_actor": selected_cfg["risk_actor"],
        "rebound_actor": selected_cfg["rebound_actor"],
        "base_position": frame[selected_cfg["base_actor"]],
        "risk_actor_position": frame[selected_cfg["risk_actor"]],
        "rebound_actor_position": frame[selected_cfg["rebound_actor"]],
        "rebound_state": states["rebound"],
        "risk_state": states["risk"],
        "high_vol_state": states["high_vol"],
        "downtrend_state": states["downtrend"],
        "vix_stress_state": states["vix_stress"],
        "semis_stress_state": states["semis_stress"],
        "soxl_mom_10": frame["soxl_mom_10"],
        "soxl_mom_21": frame["soxl_mom_21"],
        "soxl_drawdown_63": frame["soxl_drawdown_63"],
        "realized_vol_pct_21": frame["realized_vol_pct_21"],
        "vix_percentile_252": frame["vix_percentile_252"],
        "soxx_mom_21": frame["soxx_mom_21"],
        "soxl_soxx_ratio_z_252": frame["soxl_soxx_ratio_z_252"],
        "soxl_soxx_ratio_widen_21": frame["soxl_soxx_ratio_widen_21"],
    })
    out_name = "selected_strict_oos_daily.csv" if window == "strict_oos" else f"selected_{window}_daily.csv"
    out.to_csv(CARD / out_name, index=False)

selected_metrics_df = pd.DataFrame(selected_metrics_rows)
selected_metrics_df.to_csv(CARD / "selected_metrics.csv", index=False)

strict_position, _ = selector_position(strict_frame, selected_cfg)
base_cluster = base_cluster_cache[selected_cfg["base_actor"]]
cluster_rows = []
for name, (start, end) in POOR_CLUSTERS.items():
    m = sub_metrics(strict_frame, strict_position, start, end)
    b = base_cluster[name]
    cluster_rows.append({
        "cluster": name,
        "start": start,
        "end": end,
        **{f"base_actor_{k}": v for k, v in b.items()},
        **{f"selected_{k}": v for k, v in m.items()},
        "excess_delta_vs_base_actor": m["relative_return"] - b["relative_return"],
        "return_delta_vs_base_actor": m["return"] - b["return"],
    })
cluster_df = pd.DataFrame(cluster_rows)
cluster_df.to_csv(CARD / "selected_cluster_comparison.csv", index=False)

display(Markdown("## Selected metrics"))
display(selected_metrics_df)
display(Markdown("## Selected cluster comparison"))
display(cluster_df)


## Selected metrics

,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,turnover,rows,high_vol_days,downtrend_days,vix_stress_days,semis_stress_days,rebound_days,risk_days,changed_vs_base_days
0,strict_oos,1667.806381,816.512500,851.293881,1.151077,43.534970,0.449886,31.175290,1605,618,525,386,585,68,671,0
1,wf_2020,69.448916,58.093571,11.355346,1.186122,28.638902,0.387232,3.895391,252,119,99,60,58,12,119,0
2,wf_2022,-47.263003,-86.588072,39.325069,-1.356866,49.774907,0.273316,4.082471,250,156,153,128,159,4,190,0
3,wf_2023_2024,218.937254,198.504290,20.432964,1.516990,30.714702,0.468580,9.652261,501,127,116,87,192,11,159,0


## Selected cluster comparison

,cluster,start,end,base_actor_return,base_actor_buy_and_hold_return,base_actor_relative_return,base_actor_sharpe,base_actor_max_dd,base_actor_avg_pos,base_actor_turnover,base_actor_rows,selected_return,selected_buy_and_hold_return,selected_relative_return,selected_sharpe,selected_max_dd,selected_avg_pos,selected_turnover,selected_rows,excess_delta_vs_base_actor,return_delta_vs_base_actor
0,cluster_2025_high_vol_rebound,2025-04-16,2025-07-17,80.099055,192.505345,-112.406290,5.464897,10.131948,0.563993,0.823438,63,80.099055,192.505345,-112.406290,5.464897,10.131948,0.563993,0.823438,63,0.0,0.0
1,cluster_2026_high_vol_rebound,2026-02-17,2026-05-15,4.021956,155.135970,-151.114014,2.152755,1.119098,0.048717,0.702077,63,4.021956,155.135970,-151.114014,2.152755,1.119098,0.048717,0.702077,63,0.0,0.0


In [6]:
def metric_record(window: str) -> dict:
    row = selected_metrics_df.loc[selected_metrics_df["window"] == window].iloc[0].to_dict()
    return {k: (float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v) for k, v in row.items() if k != "window"}


def blockers_for(row: dict) -> list[str]:
    blockers = []
    if row["strict_oos_relative_return"] < STRICT_TARGET_REL:
        blockers.append(f"strict relative {row['strict_oos_relative_return']:.2f}pp < {STRICT_TARGET_REL:.2f}pp")
    if row["strict_oos_max_dd"] > STRICT_TARGET_DD:
        blockers.append(f"strict MaxDD {row['strict_oos_max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
    for wf, floor in WF_REL_FLOORS.items():
        if row[f"{wf}_relative_return"] < floor:
            blockers.append(f"{wf} relative {row[f'{wf}_relative_return']:.2f}pp < {floor:.2f}pp")
        if row[f"{wf}_max_dd"] > STRICT_TARGET_DD:
            blockers.append(f"{wf} MaxDD {row[f'{wf}_max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
    for _, c in cluster_df.iterrows():
        if c["selected_max_dd"] > STRICT_TARGET_DD:
            blockers.append(f"{c['cluster']} MaxDD {c['selected_max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
        if c["excess_delta_vs_base_actor"] < 0:
            blockers.append(f"{c['cluster']} excess delta {c['excess_delta_vs_base_actor']:.2f}pp < 0")
    return blockers

strict_pass_count = int(screen_df["strict_pass"].sum())
wf_pass_count = int(screen_df["wf_pass"].sum())
full_pass_count = int(screen_df["full_pass"].sum())
cluster_pass = bool((cluster_df["excess_delta_vs_base_actor"] >= 0.0).all() and (cluster_df["selected_max_dd"] <= STRICT_TARGET_DD).all())
blockers = blockers_for(selected)
best_changed = changed_screen_df.iloc[0].to_dict() if not changed_screen_df.empty else None


if full_pass_count > 0:
    decision = "diagnostic_candidate_for_training_followup"
    conclusion = [
        "At least one state-conditioned actor selector clears the full strict, walk-forward, and cluster gate.",
        "Do not promote directly from this notebook; implement a governed selector or train a state-conditioned actor and rerun promotion evidence.",
    ]
else:
    decision = "diagnostic_no_existing_actor_selector_pass"
    conclusion = [
        "No compact state-conditioned selector over existing all-WF actor exports clears the strict plus walk-forward plus cluster gate.",
        "This blocks the idea that existing actors only need a model-selection router.",
        "The next valid step is training-time state-conditioned objective / actor refresh with walk-forward selection, not another actor-selection threshold sweep.",
        "Primary blockers: " + "; ".join(blockers[:8]),
    ]

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "changed_layer": ["model_selection", "composition"],
    "source_card": str(SOURCE_CARD.relative_to(ROOT)),
    "profile_pool": profile_names,
    "candidate_count": int(len(screen_df)),
    "strict_pass_count": strict_pass_count,
    "wf_pass_count": wf_pass_count,
    "full_pass_count": full_pass_count,
    "cluster_pass": cluster_pass,
    "selected_candidate_id": selected_candidate_id,
    "selected_config": selected_cfg,
    "selected_metrics": {window: metric_record(window) for window in WINDOWS},
    "selected_cluster_comparison": cluster_df.to_dict("records"),
    "selected_screen_row": {k: (float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v) for k, v in selected.items()},
    "changed_candidate_count": int(len(changed_screen_df)),
    "best_changed_screen_row": None if best_changed is None else {k: (float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v) for k, v in best_changed.items()},
    "acceptance": {
        "strict_relative_target_pp": STRICT_TARGET_REL,
        "strict_max_dd_target_pct": STRICT_TARGET_DD,
        "strict_pass": bool(selected["strict_pass"]),
        "wf_pass": bool(selected["wf_pass"]),
        "full_pass": bool(selected["full_pass"]),
        "cluster_pass": cluster_pass,
    },
    "external_sources": [
        "https://doi.org/10.21314/JCF.2016.322",
        "https://doi.org/10.3233/AF-180231",
        "https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf",
        "https://github.com/microsoft/qlib",
    ],
    "conclusion": conclusion,
    "artifacts": {
        "notebook": str((CARD / f"{EXPERIMENT}.ipynb").relative_to(ROOT)),
        "actor_pool_metrics": str((CARD / "actor_pool_metrics.csv").relative_to(ROOT)),
        "actor_selector_screen": str((CARD / "actor_selector_screen.csv").relative_to(ROOT)),
        "actor_selector_top_configs": str((CARD / "actor_selector_top_configs.csv").relative_to(ROOT)),
        "actor_selector_top_changed_configs": str((CARD / "actor_selector_top_changed_configs.csv").relative_to(ROOT)),
        "actor_selector_family_summary": str((CARD / "actor_selector_family_summary.csv").relative_to(ROOT)),
        "selected_metrics": str((CARD / "selected_metrics.csv").relative_to(ROOT)),
        "selected_cluster_comparison": str((CARD / "selected_cluster_comparison.csv").relative_to(ROOT)),
        "selected_daily": {
            "strict_oos": str((CARD / "selected_strict_oos_daily.csv").relative_to(ROOT)),
            "wf_2020": str((CARD / "selected_wf_2020_daily.csv").relative_to(ROOT)),
            "wf_2022": str((CARD / "selected_wf_2022_daily.csv").relative_to(ROOT)),
            "wf_2023_2024": str((CARD / "selected_wf_2023_2024_daily.csv").relative_to(ROOT)),
        },
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
    },
}
with (CARD / "selection.json").open("w", encoding="utf-8") as f:
    json.dump(selection, f, indent=2, ensure_ascii=False)

print(json.dumps({
    "decision": decision,
    "candidate_count": len(screen_df),
    "selected_candidate_id": selected_candidate_id,
    "strict_pass_count": strict_pass_count,
    "wf_pass_count": wf_pass_count,
    "full_pass_count": full_pass_count,
}, indent=2))


{
  "decision": "diagnostic_no_existing_actor_selector_pass",
  "candidate_count": 27648,
  "selected_candidate_id": 16512,
  "strict_pass_count": 0,
  "wf_pass_count": 0,
  "full_pass_count": 0
}


In [7]:
strict = selection["selected_metrics"]["strict_oos"]
wf2022 = selection["selected_metrics"]["wf_2022"]
wf2324 = selection["selected_metrics"]["wf_2023_2024"]
cluster_2025, cluster_2026 = selection["selected_cluster_comparison"]
best_changed = selection.get("best_changed_screen_row")
if best_changed is None:
    best_changed_text = "No actor-switching candidate was generated."
else:
    best_changed_text = (
        f"Best changed selector: base `{best_changed['base_actor']}`, risk `{best_changed['risk_actor']}`, "
        f"rebound `{best_changed['rebound_actor']}`, strict relative `{best_changed['strict_oos_relative_return']:.2f}pp`, "
        f"strict MaxDD `{best_changed['strict_oos_max_dd']:.2f}%`, wf_2022 MaxDD `{best_changed['wf_2022_max_dd']:.2f}%`."
    )
run_conclusion = f"""
## Run conclusion

- Decision: `{selection['decision']}`; no runtime promotion.
- Screened candidates: `{selection['candidate_count']}` compact all-WF actor selectors; strict pass `{selection['strict_pass_count']}`, walk-forward pass `{selection['wf_pass_count']}`, full pass `{selection['full_pass_count']}`.
- Selected actor mix: base `{selection['selected_config']['base_actor']}`, risk `{selection['selected_config']['risk_actor']}`, rebound `{selection['selected_config']['rebound_actor']}`.
- Strict OOS: return `{strict['return']:.2f}%`, buy-hold `{strict['buy_and_hold_return']:.2f}%`, relative `{strict['relative_return']:.2f}pp`, MaxDD `{strict['max_dd']:.2f}%`, avg position `{strict['avg_pos']:.3f}`.
- Walk-forward blockers: `wf_2022` relative `{wf2022['relative_return']:.2f}pp`, MaxDD `{wf2022['max_dd']:.2f}%`; `wf_2023_2024` relative `{wf2324['relative_return']:.2f}pp`, MaxDD `{wf2324['max_dd']:.2f}%`.
- Poor clusters vs selected base actor: 2025 excess delta `{cluster_2025['excess_delta_vs_base_actor']:.2f}pp`, selected MaxDD `{cluster_2025['selected_max_dd']:.2f}%`; 2026 excess delta `{cluster_2026['excess_delta_vs_base_actor']:.2f}pp`, selected MaxDD `{cluster_2026['selected_max_dd']:.2f}%`.
- {best_changed_text}
- Interpretation: existing all-WF actors do not contain a selector-composable path that satisfies `+30pp / <=15%`. The next valid experiment should alter training-time objectives, actor generation, or state-conditioned checkpoint weighting rather than continue actor-selection threshold sweeps.
"""
display(Markdown(run_conclusion))



## Run conclusion

- Decision: `diagnostic_no_existing_actor_selector_pass`; no runtime promotion.
- Screened candidates: `27648` compact all-WF actor selectors; strict pass `0`, walk-forward pass `0`, full pass `0`.
- Selected actor mix: base `balanced_rule_residual_v19_reg`, risk `balanced_rule_residual_v19_reg`, rebound `balanced_rule_residual_v19_reg`.
- Strict OOS: return `1667.81%`, buy-hold `816.51%`, relative `851.29pp`, MaxDD `43.53%`, avg position `0.450`.
- Walk-forward blockers: `wf_2022` relative `39.33pp`, MaxDD `49.77%`; `wf_2023_2024` relative `20.43pp`, MaxDD `30.71%`.
- Poor clusters vs selected base actor: 2025 excess delta `0.00pp`, selected MaxDD `10.13%`; 2026 excess delta `0.00pp`, selected MaxDD `1.12%`.
- Best changed selector: base `balanced`, risk `balanced_rule_residual_v19_reg`, rebound `balanced_rule_residual_mode_v1`, strict relative `1283.02pp`, strict MaxDD `43.30%`, wf_2022 MaxDD `49.61%`.
- Interpretation: existing all-WF actors do not contain a selector-composable path that satisfies `+30pp / <=15%`. The next valid experiment should alter training-time objectives, actor generation, or state-conditioned checkpoint weighting rather than continue actor-selection threshold sweeps.
